In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#https://www.kaggle.com/datasets/lodetomasi1995/income-classification

In [ ]:
df = pd.read_csv('14-income_evaluation.csv')

In [ ]:
# EDA & Preparing the Data

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df[" income"].value_counts()

In [ ]:
df.columns

In [ ]:
col_names = ["age", "workclass", "finalweight", "education", "education-num", "marital-status", "occupation", "relationship", "race", "sex",
             "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

In [ ]:
df.columns = col_names

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
categorical = [col for col in df.columns if df[col].dtype == "O"]
numerical = [col for col in df.columns if df[col].dtype != "O"]

In [ ]:
## Feature Separation

# We separated the features into two groups:  
# Categorical: columns with object (string) data type  
# Numerical: columns with numerical data type

In [ ]:
categorical

In [ ]:
numerical

In [ ]:
df[categorical].head()

In [ ]:
for col in categorical:
    print(df[col].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax = sns.countplot(x="income", hue="sex", data=df)
ax.set_title("Distribution of income with gender")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax = sns.countplot(x="income", hue="race", data=df)
ax.set_title("Distribution of income with race")
plt.show()

In [ ]:
sns.catplot(x=df["hours-per-week"], hue=df["income"])
plt.show()

In [ ]:
categorical

In [ ]:
df['workclass'].unique()

In [ ]:
df["workclass"].value_counts()

In [ ]:
df["workclass"] = df["workclass"].replace(" ?", np.nan)

In [ ]:
df["workclass"].value_counts()

In [ ]:
df["education"].unique()

In [ ]:
df["marital-status"].unique()

In [ ]:
df["occupation"].unique()

In [ ]:
df["occupation"] = df["occupation"].replace(" ?", np.nan)

In [ ]:
df["occupation"].value_counts()

In [ ]:
df["relationship"].unique()

In [ ]:
df["sex"].unique()

In [ ]:
df["race"].unique()

In [ ]:
df["native-country"].unique()

In [ ]:
df["native-country"] = df["native-country"].replace(" ?", np.nan)

In [ ]:
df["native-country"].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
sns.displot(df['age'], color='red')
plt.show()

In [ ]:
sns.pairplot(df, hue="income")
plt.show()

In [ ]:
# Data Splitting

In [ ]:
X = df.drop('income' ,axis=1)
y = df['income']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [ ]:
categorical = [col for col in X_train.columns if X_train[col].dtype == "O"]

In [ ]:
X_train[categorical].isnull().sum()

In [ ]:
X_test[categorical].isnull().sum()

In [ ]:
## Handling Missing Values

# Previously, we replaced "?" values with NaN.  
# Now, we fill these missing values in categorical features with their mode.  
# This prevents losing thousands of rows from the dataset.

In [ ]:
for i in [X_train, X_test]:
    i['workclass'] = i['workclass'].fillna(X_train['workclass'].mode()[0])
    i['occupation'] = i['occupation'].fillna(X_train['occupation'].mode()[0])
    i['native-country'] = i['native-country'].fillna(X_train['native-country'].mode()[0])

In [ ]:
X_test[categorical].isnull().sum()

In [ ]:
X_test[categorical].isnull().sum()

In [ ]:
## Encoding

In [ ]:
X_train[categorical].head()

In [ ]:
df[categorical].nunique()

In [ ]:
## Encoding and Target Definition

# The dataset contains a high-cardinality feature ("native-country") with 41 unique values.  
# Instead of applying One-Hot Encoding, we define our target variable as:  
# - "1" → income > 50K  
# - "0" → income <= 50K  

# This reduces complexity and keeps the model efficient.

In [ ]:
y_train

In [ ]:
y_train_binary = y_train.apply(lambda x : 1 if x.strip() == '>50K' else 0)

In [ ]:
y_train_binary

In [ ]:
target_means = y_train_binary.groupby(X_train['native-country']).mean()

In [ ]:
target_means

In [ ]:
# ## Country-wise Income Analysis

# We calculated the mean income (binary target) for each country in the training set.  
# This shows which countries have a higher proportion of people earning more than 50K.

In [ ]:
X_train['native-country-encoded'] = X_train['native-country'].map(target_means)
X_train['native-country-encoded'] = X_train['native-country-encoded'].fillna(y_train_binary.mean())

X_test['native-country-encoded'] = X_test['native-country'].map(target_means)
X_test['native-country-encoded'] = X_test['native-country-encoded'].fillna(y_train_binary.mean())

In [ ]:
X_train.head()

In [ ]:
X_train = X_train.drop('native-country',axis=1)
X_test = X_test.drop('native-country',axis=1)

In [ ]:
categorical

In [ ]:
one_hot_categories = ['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex']

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
encoder = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown="ignore", sparse_output=False), one_hot_categories) # 1-) 'cat' ->  name , 2-) process , 3-) columns
    ], remainder="passthrough"
)

In [ ]:
X_train_enc = encoder.fit_transform(X_train)
X_test_enc = encoder.transform(X_test)

In [ ]:
X_train_enc

In [ ]:
columns = encoder.get_feature_names_out()

In [ ]:
columns

In [ ]:
X_train = pd.DataFrame(X_train_enc, columns=columns, index=X_train.index)
X_test = pd.DataFrame(X_test_enc, columns=columns, index=X_test.index)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
cols = X_train.columns

In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()

In [ ]:
# we don't actually need scaling in dt algortihms but let's do it because we will introduce robust scaler
# robust scaler is different than standardscaler and it is designed to handle outliers.

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# we don't have to convert X_train into df again, we can just give it as a numpy array to model but 
# if we want to use column names again later it will come in handy

X_train = pd.DataFrame(X_train, columns=cols)
X_test = pd.DataFrame(X_test, columns=cols)

In [ ]:
X_train

In [ ]:
# Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(n_estimators=10, random_state=42)

In [ ]:
rfc.fit(X_train,y_train)

In [ ]:
y_pred = rfc.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
print("accuracy_score: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))
print("confusion_matrix: \n", confusion_matrix(y_test,y_pred))

In [ ]:
# let's try to see feature importance

In [ ]:
rfc.feature_importances_

In [ ]:
feature_scores = pd.Series(rfc.feature_importances_, index = X_train.columns).sort_values(ascending=False)

In [ ]:
feature_scores

In [ ]:
feature_scores.tail(10)

In [ ]:
# drop the least important 10 columns from train, test and try again
# generally you do not need to do this manually but if you think you somehow made too many columns with encoding
# it's worth a try

In [ ]:
X_train = X_train.drop(["cat__education_ 12th","cat__marital-status_ Married-AF-spouse","cat__marital-status_ Married-spouse-absent",
                        "cat__education_ 5th-6th","cat__education_ 1st-4th","cat__workclass_ Without-pay","cat__occupation_ Priv-house-serv",
                        "cat__education_ Preschool","cat__occupation_ Armed-Forces","cat__workclass_ Never-worked"
                       ], axis=1)

In [ ]:
X_test = X_test.drop(["cat__education_ 12th","cat__marital-status_ Married-AF-spouse","cat__marital-status_ Married-spouse-absent",
                        "cat__education_ 5th-6th","cat__education_ 1st-4th","cat__workclass_ Without-pay","cat__occupation_ Priv-house-serv",
                        "cat__education_ Preschool","cat__occupation_ Armed-Forces","cat__workclass_ Never-worked"
                       ], axis=1)

In [ ]:
print("accuracy_score: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))
print("confusion_matrix: \n", confusion_matrix(y_test,y_pred))

In [ ]:
# We removed 10 features with low importance values.  
# The accuracy did not change significantly, indicating that these features had little or no impact on the model’s performance.

In [ ]:
## Hyperparameter tuning

In [ ]:
rf_params = {
    "n_estimators": [100,200,500,1000],
    "max_depth": [5,8,10,15,None],
    "max_features": ["sqrt", "log2", 5,6,7,8],
    "min_samples_split": [2, 8, 15, 20]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
rfc = RandomForestClassifier()

In [ ]:
rscv = RandomizedSearchCV(estimator=rfc, param_distributions=rf_params, cv=3, n_jobs=-1)
rscv.fit(X_train,y_train)

In [ ]:
y_pred = rscv.predict(X_test)

print("accuracy_score: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))
print("confusion_matrix: \n", confusion_matrix(y_test,y_pred))

In [ ]:
# accuracy: 0.8461 ----> 0.8652

In [ ]:
rscv.best_params_